In [0]:
-- ============================================
-- DIMENSION TABLES
-- Source: 02-clean
-- ============================================

-- 1. DIM_STUDENT_ENROLLMENT
-- Grain: one row per student per course presentation

CREATE OR REPLACE TABLE `ftw-week-07`.`03-mart`.dim_student_enrollment AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY
            si.id_student,
            si.code_module,
            si.code_presentation
    ) AS student_enrollment_key,

    si.id_student,
    si.code_module,
    si.code_presentation,
    si.gender,
    si.region,
    si.highest_education,
    si.imd_band,
    si.age_band,
    si.num_of_prev_attempts,
    si.studied_credits,
    si.disability,
    si.final_result,
    sr.date_registration,
    sr.date_unregistration

FROM `ftw-week-07`.`02-clean`.student_info si

LEFT JOIN `ftw-week-07`.`02-clean`.student_registration sr
    ON si.id_student = sr.id_student
    AND si.code_module = sr.code_module
    AND si.code_presentation = sr.code_presentation;


-- 2. DIM_ASSESSMENT
-- Grain: one row per assessment

CREATE OR REPLACE TABLE `ftw-week-07`.`03-mart`.dim_assessment AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY a.id_assessment
    ) AS assessment_key,

    a.id_assessment,
    a.code_module,
    a.code_presentation,
    a.assessment_type,
    a.due_day_offset,
    a.weight

FROM `ftw-week-07`.`02-clean`.assessments a;


-- 3. DIM_SITE
-- Grain: one row per VLE site/activity

CREATE OR REPLACE TABLE `ftw-week-07`.`03-mart`.dim_site AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY v.id_site
    ) AS site_key,

    v.id_site,
    v.code_module,
    v.code_presentation,
    v.activity_type,
    v.week_from,
    v.week_to

FROM `ftw-week-07`.`02-clean`.vle v;

In [0]:
-- 5. FACT_ACTIVITY
-- Grain: one row per student enrollment per VLE site per activity date
-- Source: student_vle
CREATE OR REPLACE TABLE `ftw-week-07`.`03-mart`.fact_activity AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY
            se.student_enrollment_key,
            ds.site_key,
            sv.date
    ) AS fact_activity_key,
    se.student_enrollment_key,
    ds.site_key,
    sv.date       AS activity_date,
    sv.sum_click  AS total_clicks
FROM `ftw-week-07`.`02-clean`.student_vle sv
LEFT JOIN `ftw-week-07`.`03-mart`.dim_student_enrollment se
    ON sv.id_student        = se.id_student
    AND sv.code_module       = se.code_module
    AND sv.code_presentation = se.code_presentation
LEFT JOIN `ftw-week-07`.`03-mart`.dim_site ds
    ON sv.id_site            = ds.id_site
    AND sv.code_module       = ds.code_module
    AND sv.code_presentation = ds.code_presentation;


In [0]:
-- 4. FACT_ASSESSMENT
-- Grain: one row per student enrollment per assessment
-- Source: student_assessment (has no code_module/code_presentation of its own —
-- conformed via assessments, per the bronze comment)
CREATE OR REPLACE TABLE `ftw-week-07`.`03-mart`.fact_assessment AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY
            se.student_enrollment_key,
            da.assessment_key
    ) AS fact_assessment_key,
    se.student_enrollment_key,   -- FK, nullable: see note on banked assessments below
    da.assessment_key,
    sa.date_submitted,
    sa.is_banked,
    sa.score
FROM `ftw-week-07`.`02-clean`.student_assessment sa
LEFT JOIN `ftw-week-07`.`03-mart`.dim_assessment da
    ON sa.id_assessment = da.id_assessment
LEFT JOIN `ftw-week-07`.`03-mart`.dim_student_enrollment se
    ON sa.id_student        = se.id_student
    AND da.code_module       = se.code_module
    AND da.code_presentation = se.code_presentation;